### FORECASTING VENTAS_2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl
import streamlit as st
import holidays as hdays

### CARGA DE DATOS DE INFERENCIA

In [2]:
from pathlib import Path

base_dir = Path.cwd()
project_dir = base_dir.parent if base_dir.name == "notebooks" else base_dir
inferencia_path = project_dir / "data" / "raw" / "inferencia" / "ventas_2025_inferencia.csv"

inferencia_df = pd.read_csv(inferencia_path)
inferencia_df.head()

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [3]:
from pathlib import Path

# modelo_final se entrena con df y sus columnas one-hot, no con df_model,
# que contiene las variables temporales y los lags construidos aparte.
columnas_categoricas = ["nombre", "categoria", "subcategoria"]
columnas_categoricas_h = [f"{columna}_h" for columna in columnas_categoricas]
columnas_excluidas = {
    "fecha", "ingresos", "unidades_vendidas",
    "nombre", "producto_id", "categoria", "subcategoria",
}

# Alinear con el esquema guardado del df usado durante el entrenamiento.
training_schema_path = project_dir / "data" / "processed" / "df.csv"
if not training_schema_path.exists():
    raise FileNotFoundError(
        f"No se encontro el esquema de entrenamiento: {training_schema_path}"
    )
columnas_entrenamiento = pd.read_csv(training_schema_path, nrows=0).columns.tolist()

prefijos_one_hot = tuple(f"{columna_h}_" for columna_h in columnas_categoricas_h)
columnas_base_esperadas = [
    columna for columna in columnas_entrenamiento
    if not columna.startswith(prefijos_one_hot)
]
columnas_base_faltantes = [
    columna for columna in columnas_base_esperadas if columna not in inferencia_df.columns
]
if columnas_base_faltantes:
    raise KeyError(
        f"Faltan columnas requeridas por el modelo: {columnas_base_faltantes}"
    )

inferencia_df["fecha"] = pd.to_datetime(inferencia_df["fecha"], errors="raise")
for columna, columna_h in zip(columnas_categoricas, columnas_categoricas_h):
    inferencia_df[columna_h] = inferencia_df[columna]

inferencia_df = pd.get_dummies(
    inferencia_df,
    columns=columnas_categoricas_h,
    prefix=columnas_categoricas_h,
    prefix_sep="_",
    drop_first=False,
)

columnas_one_hot_esperadas = [
    columna for columna in columnas_entrenamiento
    if columna.startswith(prefijos_one_hot)
]
columnas_one_hot_generadas = [
    columna for columna in inferencia_df.columns
    if columna.startswith(prefijos_one_hot)
]
categorias_no_vistas = sorted(
    set(columnas_one_hot_generadas) - set(columnas_one_hot_esperadas)
)
if categorias_no_vistas:
    raise ValueError(
        f"Categorias de inferencia ausentes en entrenamiento: {categorias_no_vistas}"
    )

for columna in columnas_one_hot_esperadas:
    if columna not in inferencia_df.columns:
        inferencia_df[columna] = False

# Reutilizar orden y columnas de df, para que coincidan con modelo_final.
inferencia_df = inferencia_df.reindex(columns=columnas_entrenamiento)

# Aplicar el filtro temporal despues de completar todas las transformaciones.
inferencia_df = inferencia_df.loc[
    inferencia_df["fecha"].dt.year.eq(2025)
    & inferencia_df["fecha"].dt.month.eq(11)
].copy()
if inferencia_df.empty:
    raise ValueError("No hay registros de noviembre de 2025 para inferencia.")

columnas_predictoras = [
    columna for columna in columnas_entrenamiento
    if columna not in columnas_excluidas
]
inferencia_X = inferencia_df[columnas_predictoras].copy()

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "inferencia_df_transformado.csv"
inferencia_df.to_csv(output_path, index=False)

print(f"Registros guardados (solo noviembre de 2025): {len(inferencia_df):,}")
print(f"Estructura de inferencia_df: {inferencia_df.shape}")
print(f"Variables de entrada para modelo_final: {inferencia_X.shape[1]}")
print(f"Archivo guardado en: {output_path.resolve()}")
display(inferencia_df.head())

Registros guardados (solo noviembre de 2025): 720
Estructura de inferencia_df: (720, 57)
Variables de entrada para modelo_final: 50
Archivo guardado en: C:\Users\451-9610\Documents\Otros\Cursos\2026\DS4B\Dia_1\ForecastingVentas\data\processed\inferencia_df_transformado.csv


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,False,False,False,False,False,False,True,False
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,False,False,False,False,False,False,True,False
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,False,False,False,False,False,False,True,False
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,False,False,False,False,False,True,False,False


In [6]:
inferencia_df.shape

(720, 57)

In [7]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'Amazon', 'Decathlon', 'Deporvillage',
       'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23',
       'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552',
       'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900',
       'nombre_h_Domyos Kit Mancuernas 20kg',
       'nombre_h_Gaiam Premium Yoga Block', 'nombre_h_Liforme Yoga Pad',
       'nombre_h_Lotuscrafts Yoga Bolster', 'nombre_h_Manduka PRO Yoga Mat',
       'nombre_h_Merrell Moab 2 GTX',
       'nombre_h_New Balance Fresh Foam X 1080v12',
       'nombre_h_Nike Air Zoom Pegasus 40', 'nombre_h_Nike Dri-FIT Miler',
       'nombre_h_Puma Velocity Nitro 2', 'nombre_h_Quechua MH500',
       'nombre_h_Reebok Floatride Energy 5',
       'nombre_h_Reebok Professional Deck',
       'nombr